# Step 06: Comprehensive Clean Test Set Evaluation
### Benchmark Evaluation: COCO mAP@50:95, Multi-Threshold Sweeps, Confusion Matrix & Error Inspector
Directly extracted from `multi_class_train_rfdetr.ipynb` (Cells 1, 2, 14, 15, 16, 17, 19).

### Metrics Computed:
- Official COCO Metrics: **mAP@50:95**, **mAP@50**, **mAP@75**, **mAR@100**.
- Multi-Threshold Sweeps: Precision, Recall, and F1 across confidence levels [0.10 to 0.90].
- Multi-Class Confusion Matrix across all auto-discovered categories.
- 100% Visual Error Inspector: Isolates False Positives (hallucinated tags) and False Negatives (missed tags).

In [ ]:
# STEP 0 - Environment Dependencies
!pip uninstall -y opencv-python opencv-contrib-python opencv-python-headless
!pip install --no-cache-dir opencv-python-headless==4.9.0.80
%pip install python-dotenv

In [ ]:
# CELL 1 - Imports, VRAM Monitor & Logger Initialization
import os, sys, json, time, math, copy, logging, random, shutil, warnings
warnings.filterwarnings("ignore", message=".*meshgrid.*")
warnings.filterwarnings("ignore", message=".*torch.cuda.amp.autocast.*")
warnings.filterwarnings("ignore", message=".*max_detection_threshold.*")
warnings.filterwarnings("ignore", category=FutureWarning)
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple
from collections import defaultdict, Counter
import urllib.parse
from urllib.parse import urlparse, unquote
from concurrent.futures import ThreadPoolExecutor, as_completed

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(line_buffering=True)
os.environ["PYTHONUNBUFFERED"] = "1"

# Configure multiprocessing sharing strategy to prevent 'Too many open files' error
import torch.multiprocessing as mp
try:
    mp.set_sharing_strategy('file_system')
except Exception:
    pass

try:
    import resource
    rlimit = resource.getrlimit(resource.RLIMIT_NOFILE)
    resource.setrlimit(resource.RLIMIT_NOFILE, (max(rlimit[0], 4096), max(rlimit[1], 4096)))
except Exception:
    pass

import cv2, torch, requests, supervision as sv
# Neutralize Matplotlib interactive callback to permanently prevent _draw_all_if_interactive crash
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.ioff()
plt.close('all')
try:
    _ip = get_ipython()
    if _ip and hasattr(_ip, 'events'):
        _ip.events.callbacks['post_execute'] = [
            _cb for _cb in _ip.events.callbacks.get('post_execute', [])
            if 'draw_all' not in getattr(_cb, '__name__', '')
        ]
except Exception:
    pass
import matplotlib.patches as patches, numpy as np, pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from PIL import Image, ImageDraw, ImageFont
from dotenv import load_dotenv
from tqdm.auto import tqdm

try:
    from torchmetrics.detection.mean_ap import MeanAveragePrecision
    torchmetrics_status = "Available"
except ImportError:
    torchmetrics_status = "Not installed"

try:
    from rfdetr import RFDETRNano, RFDETRSmall, RFDETRMedium, RFDETRBase, RFDETRLarge
    from rfdetr import main as rfdetr_main
    from rfdetr.models.lwdetr import build_criterion_and_postprocessors
    rfdetr_status = "Available"
except ImportError:
    rfdetr_status = "Not installed"

# Auto-flushing console and file logger
class FlushHandler(logging.StreamHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

class FlushFileHandler(logging.FileHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

logger = logging.getLogger("rfdetr_multi_class")
logger.setLevel(logging.INFO)
logger.handlers.clear()
logger.propagate = False
log_format = "%(asctime)s | %(levelname)-8s | %(message)s"
console_handler = FlushHandler(sys.stdout)
console_handler.setFormatter(logging.Formatter(log_format))
logger.addHandler(console_handler)

def print_vram_usage(tag="Status"):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / (1024**3)
        total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        logger.info(f"[VRAM - {tag}] {torch.cuda.get_device_name(0)}: {alloc:.2f} GB / {total:.2f} GB allocated")
    else:
        logger.info(f"[VRAM - {tag}] Running on CPU")

# Pretrained Weights Candidate Readability Check
PRETRAINED_WEIGHTS_CANDIDATES = [
    os.path.expanduser("~/rf-detr-base-coco.pth"),
    "/home/jupyter/rf-detr-base-coco.pth",
    "rf-detr-base-coco.pth"
]

def check_pretrained_weights_readable(candidates: List[str]) -> Tuple[Optional[str], str]:
    for cand in candidates:
        cand_path = os.path.expanduser(cand)
        if os.path.exists(cand_path) and os.access(cand_path, os.R_OK) and os.path.getsize(cand_path) > 1024*1024:
            try:
                ckpt = torch.load(cand_path, map_location="cpu", weights_only=False)
                count = len(ckpt.get("model", ckpt).keys()) if isinstance(ckpt, dict) else "?"
                del ckpt
                return cand_path, f"Readable ({os.path.getsize(cand_path)/1e6:.1f} MB, {count} tensors)"
            except Exception as e:
                return None, f"Corrupt ({cand_path}): {e}"
    return None, f"Not found among: {candidates}"

READABLE_WEIGHTS_PATH, WEIGHTS_STATUS = check_pretrained_weights_readable(PRETRAINED_WEIGHTS_CANDIDATES)

logger.info("=" * 65)
logger.info(f"[CELL 1] Core Libraries & Logger Ready (PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()})")
if READABLE_WEIGHTS_PATH:
    logger.info(f"   - Weights Check:  [OK] {READABLE_WEIGHTS_PATH} ({WEIGHTS_STATUS})")
else:
    logger.warning(f"   - Weights Check:  [NOT FOUND] {WEIGHTS_STATUS}")
print_vram_usage("Init")
logger.info("=" * 65)

In [ ]:
# CELL 2 - Evaluation Paths & Model Discovery
REPO_ROOT = next((p.resolve() for p in [Path("."), Path(".."), Path("../..")] if (p / "coco_files").exists() or (p / ".git").exists()), Path(".").resolve())
PIPELINE_NAME = "multi_class_rfdetr"
PIPELINE_DIR = REPO_ROOT / PIPELINE_NAME
INPUT_JSON_DIR = Path("coco_files") if Path("coco_files").exists() else (REPO_ROOT / "coco_files")
MODEL_VERSION = os.getenv("MODEL_VERSION", "v2")
VERSION_TAG = MODEL_VERSION if MODEL_VERSION else "v1"
VERSION_DIR = PIPELINE_DIR / VERSION_TAG
# Dynamic Model Weight Discovery: Automatically selects the latest or best trained model checkpoint
# 1. Look across all standard version, model, and checkpoint output directories
candidate_model_dirs = [
    VERSION_DIR / "model",
    VERSION_DIR / "checkpoints",
    PIPELINE_DIR / "model",
    PIPELINE_DIR / "checkpoints",
    REPO_ROOT / PIPELINE_NAME / "model",
    REPO_ROOT / PIPELINE_NAME / "checkpoints",
    REPO_ROOT / "model",
    REPO_ROOT / "checkpoints",
    REPO_ROOT / "multi_class_train_rfdetr" / "model",
    REPO_ROOT / "multi_class_train_rfdetr" / "checkpoints",
    Path("/home/jupyter/location_tag/multi_class_rfdetr/v2/checkpoints"),
    Path("/home/jupyter/location_tag/multi_class_rfdetr/v2/model"),
    Path("/home/jupyter/multi_class_rfdetr/v2/checkpoints"),
    Path("/home/jupyter/multi_class_rfdetr/v2/model"),
    Path("model"),
    Path("checkpoints")
]

# 2. Gather all .pth checkpoints across all valid directories
all_discovered_pths = []
for d in candidate_model_dirs:
    if d.exists():
        for p in d.glob("*.pth"):
            if p.is_file() and p.stat().st_size > 1024 * 1024:
                all_discovered_pths.append(p)

# 3. Best Model from Training Discovery & Verification
# Training exports the best model checkpoint as 'best_model.pth' and 'best_model_{RUN_ID}.pth'
# in FINAL_MODEL_DIR (e.g. multi_class_rfdetr/v2/model/) and CHECKPOINT_DIR (multi_class_rfdetr/v2/checkpoints/).
# Prioritize the true best model checkpoint from training:
BEST_MODEL_PATH = None
training_best_candidates = []

priority_best_locations = [
    VERSION_DIR / "model" / "best_model.pth",
    VERSION_DIR / "checkpoints" / "best_model.pth",
    PIPELINE_DIR / "model" / f"best_model_full_data_{VERSION_TAG}.pth",
    PIPELINE_DIR / "model" / "best_model_full_data_v2.pth",
    PIPELINE_DIR / "model" / "best_model.pth",
    PIPELINE_DIR / "checkpoints" / "best_model.pth",
    REPO_ROOT / "multi_class_train_rfdetr" / "model" / "best_model.pth",
    REPO_ROOT / "best_model.pth",
]

for p in priority_best_locations:
    if p.exists() and p.is_file() and p.stat().st_size > 1024 * 1024:
        training_best_candidates.append(p)

if training_best_candidates:
    BEST_MODEL_PATH = training_best_candidates[0]
elif all_discovered_pths:
    best_named = [p for p in all_discovered_pths if p.name == "best_model.pth"]
    version_named = [p for p in all_discovered_pths if "best_model_" in p.name]
    best_metric = [p for p in all_discovered_pths if "best_loss" in p.name.lower() or "best_map" in p.name.lower() or "precision" in p.name.lower()]
    epoch_snaps = [p for p in all_discovered_pths if "epoch_" in p.name.lower()]
    latest_candidates = [p for p in all_discovered_pths if "latest" in p.name.lower()]
    
    if best_named:
        best_named.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        BEST_MODEL_PATH = best_named[0]
    elif version_named:
        version_named.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        BEST_MODEL_PATH = version_named[0]
    elif best_metric:
        best_metric.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        BEST_MODEL_PATH = best_metric[0]
    elif epoch_snaps:
        epoch_snaps.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        BEST_MODEL_PATH = epoch_snaps[0]
    elif latest_candidates:
        BEST_MODEL_PATH = latest_candidates[0]
    else:
        all_discovered_pths.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        BEST_MODEL_PATH = all_discovered_pths[0]

FINAL_MODEL_DIR = BEST_MODEL_PATH.parent if BEST_MODEL_PATH else candidate_model_dirs[0]

candidate_dataset_dirs = [
    PIPELINE_DIR / "dataset",
    PIPELINE_DIR / "dataset_full_data",
    VERSION_DIR / "dataset",
    REPO_ROOT / "dataset",
    REPO_ROOT / "multi_class_rfdetr" / "dataset",
    Path("/home/jupyter/multi_class_rfdetr/dataset"),
    REPO_ROOT / "multi_class_train_rfdetr" / "dataset_full_data",
    Path("dataset_stratified")
]
DATASET_DIR = next((p for p in candidate_dataset_dirs if (p / "test" / "_annotations.coco.json").exists()), candidate_dataset_dirs[0])

TEST_ANN = str(DATASET_DIR / "test" / "_annotations.coco.json")
candidate_image_dirs = [
    PIPELINE_DIR / "images",
    DATASET_DIR / "images",
    DATASET_DIR / "test",
    INPUT_JSON_DIR / "images",
    REPO_ROOT / "coco_files" / "images",
    REPO_ROOT / "images",
    Path("/home/jupyter/location_tag/coco_files/images"),
    Path("/home/jupyter/location_tag/images"),
    Path("/home/jupyter/coco_files/images"),
    Path("/home/jupyter/images")
]
IMAGES_DIR = next((p for p in candidate_image_dirs if p.exists() and any(p.glob("*"))), candidate_image_dirs[0])

# =========================================================================
# EVALUATION & POSTPROCESSING CONFIGURATION (OOM-Safe Execution)
# =========================================================================
CONFIDENCE_THRESHOLD = 0.50    # Confidence threshold for test evaluation (e.g. 0.30 - 0.55)
BOX_AVERAGING_METHOD = "wbf"   # Options: "wbf" (Weighted Box Fusion), "nms" (Greedy NMS), "none" (Raw)
FUSION_IOU_THRESHOLD = 0.50    # IoU threshold for WBF / NMS box clustering
BOX_SHRINK_FACTOR = 0.96       # Margin shrinkage factor (1.0 = disabled, 0.96 = 4% contraction)
RESOLUTION = 560               # Model evaluation input resolution

# VRAM & Memory Management Settings (Guaranteed zero OOM)
EVAL_BATCH_SIZE = 1            # Batch size = 1 avoids peak activation spikes on GPU
NUM_EVAL_WORKERS = 0           # 0 workers avoids multiprocessing RAM overhead and shared memory leaks
MAX_EVAL_SAMPLES = None        # Set an integer (e.g. 500) for quick audit, or None for entire test set

# Backward compatibility aliases
CONFIDENCE = CONFIDENCE_THRESHOLD
NMS_THRESHOLD = FUSION_IOU_THRESHOLD

with open(TEST_ANN) as f:
    coco_test_data = json.load(f)

AUTO_CATEGORIES = [c["name"] for c in coco_test_data.get("categories", [])]
NUM_CLASSES = len(AUTO_CATEGORIES)
class_names = AUTO_CATEGORIES

logger.info(f"Evaluating Model:    {BEST_MODEL_PATH}")
logger.info(f"Test Annotations:    {TEST_ANN} ({len(coco_test_data.get('images', []))} images)")

In [ ]:
# CELL 14 - Safe Pretrained Weights Loading & Detection Head Patch
from rfdetr.models.lwdetr import LWDETR

# 1. Patch LWDETR.load_state_dict using torch.nn.Module.load_state_dict directly (prevents RecursionError)
def _safe_lwdetr_load_state_dict(self, state_dict, strict=True):
    """Filters out classification heads (class_embed, enc_out_class_embed) from checkpoint when num_classes differs."""
    model_state = self.state_dict()
    filtered_state_dict = {}
    mismatched = []
    
    for k, v in state_dict.items():
        clean_k = k
        if clean_k not in model_state and clean_k.startswith("model."):
            clean_k = clean_k[6:]
        if clean_k not in model_state and clean_k.startswith("module."):
            clean_k = clean_k[7:]
            
        if clean_k in model_state:
            if model_state[clean_k].shape == v.shape:
                filtered_state_dict[clean_k] = v
            else:
                mismatched.append((clean_k, tuple(v.shape), tuple(model_state[clean_k].shape)))
        elif k in model_state:
            if model_state[k].shape == v.shape:
                filtered_state_dict[k] = v
            else:
                mismatched.append((k, tuple(v.shape), tuple(model_state[k].shape)))

    if mismatched:
        logger.info(f"Notice: Filtered {len(mismatched)} classification head layers with class-count mismatch from COCO weights:")
        for name, ckpt_shape, model_shape in mismatched[:4]:
            logger.info(f"   - {name}: checkpoint {ckpt_shape} -> target model {model_shape}")
        if len(mismatched) > 4:
            logger.info(f"   - ... and {len(mismatched) - 4} more classification head tensors.")
        logger.info("Pretrained backbone, transformer, and regression heads loaded successfully.")
    
    # Call base PyTorch nn.Module.load_state_dict directly to avoid any recursion on re-runs
    return torch.nn.Module.load_state_dict(self, filtered_state_dict, strict=False)

LWDETR.load_state_dict = _safe_lwdetr_load_state_dict
logger.info("LWDETR.load_state_dict patched for safe custom num_classes loading.")

# 2. Patch reinitialize_detection_head to handle both class_embed and transformer.enc_out_class_embed
def _fixed_reinitialize(self, num_classes):
    lwdetr = self.model  # the actual LWDETR nn.Module
    device = next(lwdetr.parameters()).device
    
    # Reinitialize class_embed
    if hasattr(lwdetr, "class_embed"):
        in_feat = lwdetr.class_embed.in_features
        lwdetr.class_embed = nn.Linear(in_feat, num_classes).to(device)
        nn.init.normal_(lwdetr.class_embed.weight, std=0.01)
        nn.init.zeros_(lwdetr.class_embed.bias)
        
    # Reinitialize transformer.enc_out_class_embed
    if hasattr(lwdetr, "transformer") and hasattr(lwdetr.transformer, "enc_out_class_embed"):
        enc_heads = lwdetr.transformer.enc_out_class_embed
        if isinstance(enc_heads, nn.ModuleList):
            for i in range(len(enc_heads)):
                in_feat = enc_heads[i].in_features
                enc_heads[i] = nn.Linear(in_feat, num_classes).to(device)
                nn.init.normal_(enc_heads[i].weight, std=0.01)
                nn.init.zeros_(enc_heads[i].bias)
        elif isinstance(enc_heads, nn.Linear):
            in_feat = enc_heads.in_features
            lwdetr.transformer.enc_out_class_embed = nn.Linear(in_feat, num_classes).to(device)
            nn.init.normal_(lwdetr.transformer.enc_out_class_embed.weight, std=0.01)
            nn.init.zeros_(lwdetr.transformer.enc_out_class_embed.bias)
            
    logger.info(f"Detection heads (class_embed + enc_out_class_embed) initialized for num_classes={num_classes}")

rfdetr_main.Model.reinitialize_detection_head = _fixed_reinitialize
logger.info("reinitialize_detection_head patched.")

# 3. Model Instantiation
MODEL_SIZE = globals().get("MODEL_SIZE", os.getenv("MODEL_SIZE", "base"))
RESOLUTION = globals().get("RESOLUTION", globals().get("INTERNAL_RESOLUTION", 560))

if "class_names" not in globals() or not globals().get("class_names"):
    resolved_ann = globals().get("TEST_ANN", None)
    if resolved_ann and os.path.exists(str(resolved_ann)):
        try:
            with open(resolved_ann, "r", encoding="utf-8") as f:
                ann_data = json.load(f)
            class_names = [c["name"] for c in sorted(ann_data.get("categories", []), key=lambda x: x["id"])]
        except Exception:
            class_names = ["blue_aisle", "blue_bay", "location_tag"]
    else:
        class_names = ["blue_aisle", "blue_bay", "location_tag"]
else:
    class_names = globals().get("class_names")

NUM_CLASSES = len(class_names)

model_cls_map = {
    "nano": RFDETRNano,
    "small": RFDETRSmall,
    "medium": RFDETRMedium,
    "base": RFDETRBase,
    "large": RFDETRLarge
}
ModelClass = model_cls_map.get(str(MODEL_SIZE).lower(), RFDETRBase)
model = ModelClass(
    num_classes=NUM_CLASSES,
    pretrain_weights=None,
    resolution=RESOLUTION
)
logger.info(f"RF-DETR-{str(MODEL_SIZE).capitalize()} instantiated with resolution={RESOLUTION}, num_classes={NUM_CLASSES}")

if hasattr(model, "model") and hasattr(model.model, "reinitialize_detection_head"):
    model.model.reinitialize_detection_head(NUM_CLASSES)

# Load latest trained model weights directly into the initialized architecture
target_weights = globals().get("BEST_MODEL_PATH", None)
if target_weights and os.path.exists(str(target_weights)):
    try:
        ckpt_data = torch.load(str(target_weights), map_location="cpu", weights_only=False)
        state_dict = ckpt_data.get("model_state_dict", ckpt_data)
        model.model.model.load_state_dict(state_dict, strict=False)
        logger.info(f"✓ Successfully loaded trained model weights from: {target_weights}")
    except Exception as e:
        logger.warning(f"Could not load trained weights from {target_weights}: {e}")
else:
    logger.warning("No trained weights found on disk. Initialized with fresh detection head.")

# Extract underlying model module and assign default device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lwdetr = model.model.model if hasattr(model, "model") and hasattr(model.model, "model") else getattr(model, "model", model)

# Clear unused outer model wrapper to free CPU/GPU RAM
del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

lwdetr.to(device)
lwdetr.eval()
logger.info(f"Model underlying LWDETR initialized in eval mode on: {device}")

In [ ]:
# CELL 15 - Optimized PyTorch Dataset with Training Augmentations (Fast OpenCV C++ Decoding)
class COCODetectionDataset(Dataset):
    """Fast COCO dataset using OpenCV with multi-strategy image resolution and fallback."""
    def __init__(self, img_dir: str, ann_file: str, resolution: int = 560, is_train: bool = False, use_augmentation: bool = False):
        with open(ann_file) as f:
            data = json.load(f)
        self.img_dir = img_dir
        self.resolution = resolution
        self.is_train = is_train
        self.use_augmentation = use_augmentation and is_train
        self.cat_to_id = {c: i for i, c in enumerate(sorted(c["id"] for c in data["categories"]))}
        self.img_meta_map = {img["id"]: img for img in data.get("images", [])}
        ann_by_img: dict = {}
        for ann in data["annotations"]:
            ann_by_img.setdefault(ann["image_id"], []).append(ann)
        self.samples = [(img, ann_by_img[img["id"]]) for img in data["images"] if img["id"] in ann_by_img]

        repo_root = globals().get("REPO_ROOT", Path(".").resolve())
        ann_parent = Path(ann_file).parent if ann_file else Path(".")
        
        self.search_dirs = [
            str(self.img_dir),
            str(Path(self.img_dir).resolve()) if self.img_dir else None,
            str(ann_parent / "images"),
            str(ann_parent.parent / "images"),
            str(ann_parent / "test"),
            str(ann_parent),
            str(repo_root / "coco_files" / "images"),
            str(repo_root / "images"),
            str(repo_root / "multi_class_rfdetr" / "images"),
            str(repo_root / "multi_class_rfdetr" / "dataset" / "images"),
            str(repo_root / "multi_class_train_rfdetr" / "images"),
            str(INPUT_JSON_DIR / "images") if "INPUT_JSON_DIR" in globals() else None,
            str(Path("coco_files/images")),
            str(Path("images")),
            "/home/jupyter/location_tag/coco_files/images",
            "/home/jupyter/location_tag/images",
            "/home/jupyter/location_tag/multi_class_rfdetr/images",
            "/home/jupyter/location_tag/multi_class_rfdetr/dataset/images",
            "/home/jupyter/multi_class_rfdetr/images",
            "/home/jupyter/multi_class_rfdetr/dataset/images",
            "/home/jupyter/multi_class_train_rfdetr/images",
            "/home/jupyter/coco_files/images",
            "/home/jupyter/images"
        ]
        self.search_dirs = [d for d in self.search_dirs if d and os.path.exists(d)]

        self.disk_file_index = {}
        self.sig_to_filename = {}
        
        for d in self.search_dirs:
            try:
                for entry in os.scandir(d):
                    if entry.is_file():
                        c_name = entry.name
                        p_str = entry.path
                        self.disk_file_index[c_name] = p_str
                        self.disk_file_index[c_name.lower()] = p_str
                        stem = Path(c_name).stem
                        self.disk_file_index[stem] = p_str
                        self.disk_file_index[stem.lower()] = p_str
                        
                        if "sig=" in c_name:
                            try:
                                s_tok = c_name.split("sig=")[-1].split("&")[0]
                                if s_tok:
                                    for k in [s_tok, f"{s_tok}.jpg", s_tok.strip("="), f"{s_tok.strip('=')}.jpg"]:
                                        self.sig_to_filename[k] = c_name
                            except Exception:
                                pass
            except Exception:
                pass

        from urllib.parse import urlparse, parse_qs, unquote
        for im in data.get("images", []):
            for u_field in ["original_url", "coco_url", "azure_url", "url", "file_name"]:
                u_val = im.get(u_field)
                if u_val and ("sig=" in str(u_val) or "?" in str(u_val)):
                    try:
                        u_str = unquote(str(u_val))
                        parsed_u = urlparse(u_str)
                        c_name = Path(parsed_u.path).name
                        if c_name and not (c_name.endswith("=") or Path(c_name).stem.endswith("=")):
                            qs = parse_qs(parsed_u.query)
                            sig = qs.get("sig", [None])[0]
                            if sig:
                                for k in [sig, f"{sig}.jpg", sig.strip("="), f"{sig.strip('=')}.jpg"]:
                                    self.sig_to_filename[k] = c_name
                    except Exception:
                        pass

        self._path_cache = {}
        self._warned_missing = set()

    def _resolve_image_path(self, raw_name: str, img_id: Any = None, img_meta: Optional[dict] = None) -> str:
        cache_key = f"{raw_name}_{img_id}"
        if cache_key in self._path_cache:
            return self._path_cache[cache_key]

        im_info = img_meta or self.img_meta_map.get(img_id, {})
        import urllib.parse
        from urllib.parse import urlparse, unquote

        raw_candidates = [
            raw_name,
            im_info.get("file_name"),
            im_info.get("original_url"),
            im_info.get("coco_url"),
            im_info.get("azure_url"),
            im_info.get("url"),
            f"{img_id}.jpg" if img_id is not None else None,
            str(img_id) if img_id is not None else None,
        ]

        for raw in raw_candidates:
            if not raw: continue
            raw_s = str(raw).strip()
            if raw_s in self.sig_to_filename:
                mapped = self.sig_to_filename[raw_s]
                if mapped in self.disk_file_index:
                    resolved = self.disk_file_index[mapped]
                    self._path_cache[cache_key] = resolved
                    return resolved
            stem_s = Path(raw_s).stem
            if stem_s in self.sig_to_filename:
                mapped = self.sig_to_filename[stem_s]
                if mapped in self.disk_file_index:
                    resolved = self.disk_file_index[mapped]
                    self._path_cache[cache_key] = resolved
                    return resolved

        cleaned_candidates = []
        for raw in raw_candidates:
            if not raw: continue
            raw_s = str(raw).strip()
            unquoted = unquote(raw_s)
            clean_s = unquoted.split("?")[0].split("#")[0]
            if "&sig=" in clean_s:
                clean_s = clean_s.split("&sig=")[0]
            p_name = Path(urlparse(clean_s).path).name or Path(clean_s).name
            if not (p_name.endswith("=") or Path(p_name).stem.endswith("=")):
                cleaned_candidates.extend([p_name, p_name.lower(), f"{Path(p_name).stem}.jpg", clean_s])
            else:
                cleaned_candidates.append(f"{img_id}.jpg" if img_id is not None else "image.jpg")

        unique_candidates = []
        for c in cleaned_candidates:
            if c and c not in unique_candidates:
                unique_candidates.append(c)

        for c in unique_candidates:
            if c in self.disk_file_index:
                resolved = self.disk_file_index[c]
                self._path_cache[cache_key] = resolved
                return resolved

        for d in self.search_dirs:
            for c in unique_candidates:
                p = os.path.join(d, c)
                if os.path.exists(p) and os.path.getsize(p) > 0:
                    self._path_cache[cache_key] = p
                    return p

        best_candidate = unique_candidates[0] if unique_candidates else (f"{img_id}.jpg" if img_id else raw_name)
        fallback = os.path.join(self.img_dir, best_candidate)
        self._path_cache[cache_key] = fallback
        return fallback

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        if len(self.samples) == 0:
            raise IndexError("Dataset contains zero valid samples.")

        img_meta, anns = self.samples[idx]
        raw_name = img_meta.get("file_name") or img_meta.get("coco_url") or img_meta.get("azure_url") or f"{img_meta.get('id')}.jpg"
        img_path = self._resolve_image_path(str(raw_name), img_id=img_meta.get("id"), img_meta=img_meta)

        if not (img_path and os.path.exists(img_path)):
            if idx not in self._warned_missing:
                self._warned_missing.add(idx)
                print(f"[DATASET NOTICE] Image not found: {img_path} (ID: {img_meta.get('id')}). Falling back to next sample.")
            return self.__getitem__((idx + 1) % len(self))

        cv_img = cv2.imread(img_path)
        if cv_img is None:
            if idx not in self._warned_missing:
                self._warned_missing.add(idx)
                print(f"[DATASET NOTICE] Corrupt image {img_path}. Falling back to next sample.")
            return self.__getitem__((idx + 1) % len(self))

        orig_h, orig_w = cv_img.shape[:2]
        img = cv2.cvtColor(cv_img, cv2.COLOR_BGR2RGB)

        boxes = []
        labels = []
        for ann in anns:
            x, y, w, h = ann["bbox"]
            if (x <= 1.05 and y <= 1.05 and w <= 1.05 and h <= 1.05) and (w > 0 or h > 0) and (orig_w > 10 and orig_h > 10):
                x, y, w, h = x * orig_w, y * orig_h, w * orig_w, h * orig_h
            if w > 0 and h > 0:
                boxes.append([x, y, w, h])
                labels.append(self.cat_to_id.get(ann["category_id"], 0))

        if self.use_augmentation and len(boxes) > 0:
            if random.random() < 0.5:
                img = cv2.flip(img, 1)
                boxes = [[orig_w - (bx + bw), by, bw, bh] for bx, by, bw, bh in boxes]
            if random.random() < 0.3:
                alpha = random.uniform(0.8, 1.2)
                beta = random.randint(-15, 15)
                img = np.clip(alpha * img + beta, 0, 255).astype(np.uint8)

        img_resized = cv2.resize(img, (self.resolution, self.resolution), interpolation=cv2.INTER_LINEAR)
        img_t = torch.from_numpy(img_resized).permute(2, 0, 1).float() / 255.0

        boxes_t = torch.as_tensor(boxes, dtype=torch.float32) if len(boxes) > 0 else torch.zeros((0, 4), dtype=torch.float32)
        labels_t = torch.as_tensor(labels, dtype=torch.int64) if len(labels) > 0 else torch.zeros((0,), dtype=torch.int64)

        if len(boxes_t) > 0:
            x, y, w, h = boxes_t.unbind(-1)
            cx = (x + w / 2.0) / float(orig_w)
            cy = (y + h / 2.0) / float(orig_h)
            nw = w / float(orig_w)
            nh = h / float(orig_h)
            cx = cx.clamp(0.0, 1.0)
            cy = cy.clamp(0.0, 1.0)
            nw = nw.clamp(0.0, 1.0)
            nh = nh.clamp(0.0, 1.0)
            keep = (nw > 0.001) & (nh > 0.001)
            norm_boxes = torch.stack([cx[keep], cy[keep], nw[keep], nh[keep]], dim=-1)
            final_labels = labels_t[keep]
        else:
            norm_boxes = torch.zeros((0, 4), dtype=torch.float32)
            final_labels = torch.zeros((0,), dtype=torch.int64)

        target = {
            "boxes": norm_boxes,
            "labels": final_labels,
            "orig_size": torch.as_tensor([orig_h, orig_w], dtype=torch.int64),
            "size": torch.as_tensor([self.resolution, self.resolution], dtype=torch.int64),
            "image_id": torch.as_tensor([img_meta["id"]], dtype=torch.int64),
        }
        return img_t, target


def collate_fn(batch):
    """Batches images into a single 4D tensor and pairs target dicts in a list."""
    images = torch.stack([item[0] for item in batch])
    targets = [item[1] for item in batch]
    return images, targets

In [ ]:
# CELL 5 - Build Test DataLoader & Load Model Weights (OOM-Safe Configuration)
# Proactively clear cached CUDA memory prior to test batch evaluation
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

test_ds = COCODetectionDataset(str(IMAGES_DIR), TEST_ANN, RESOLUTION, is_train=False)

# Support subset evaluation if MAX_EVAL_SAMPLES is set
if globals().get("MAX_EVAL_SAMPLES") and len(test_ds) > MAX_EVAL_SAMPLES:
    test_ds.samples = test_ds.samples[:MAX_EVAL_SAMPLES]
    logger.info(f"Limiting evaluation to first {MAX_EVAL_SAMPLES} samples for quick audit.")

# batch_size=1 and num_workers=0 strictly avoids CUDA Out-of-Memory and host RAM spikes
test_loader = DataLoader(
    test_ds,
    batch_size=globals().get("EVAL_BATCH_SIZE", 1),
    shuffle=False,
    num_workers=globals().get("NUM_EVAL_WORKERS", 0),
    collate_fn=collate_fn,
    pin_memory=False
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lwdetr.to(device)
lwdetr.eval()

if BEST_MODEL_PATH and os.path.exists(str(BEST_MODEL_PATH)):
    # Load on CPU first to prevent GPU memory fragmentation spikes during checkpoint deserialization
    ckpt_data = torch.load(str(BEST_MODEL_PATH), map_location="cpu", weights_only=False)
    state = ckpt_data.get("model_state_dict", ckpt_data)
    lwdetr.load_state_dict(state, strict=False)
    
    epoch_info = f" (Epoch {ckpt_data['epoch']})" if isinstance(ckpt_data, dict) and 'epoch' in ckpt_data else ""
    loss_val = ckpt_data.get('best_loss', ckpt_data.get('val_loss', 'N/A')) if isinstance(ckpt_data, dict) else 'N/A'
    best_loss_info = f" | Best Val Loss: {loss_val}" if loss_val != 'N/A' else ""
    logger.info(f"✓ Verified & loaded best model weights from training: {BEST_MODEL_PATH}{epoch_info}{best_loss_info}")
    
    del ckpt_data, state
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    logger.warning("No best model checkpoint found to evaluate! Please check training directory.")

print_vram_usage("Post-Model-Load")

In [ ]:
# CELL 17 - Optimized Training Step (AMP, Accumulation) & Step A Loss Breakdown Tracking
scaler = globals().get("scaler", torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available()))

from collections import defaultdict
def run_epoch(model, loader, criterion, optimizer, device, is_train: bool, epoch: int, num_epochs: int, max_steps: Optional[int] = None):
    """Runs one training or validation epoch cleanly without noisy step-by-step logs."""
    model.train(is_train)
    phase = "train" if is_train else "val"
    total, count = 0.0, 0
    component_totals = defaultdict(float)
    total_steps = min(len(loader), max_steps) if (is_train and max_steps) else len(loader)
    pbar = tqdm(loader, total=total_steps, desc=f"Epoch {epoch:>2}/{num_epochs} [{phase}]", leave=False, unit="batch", dynamic_ncols=True)

    with torch.set_grad_enabled(is_train):
        for step, (images, targets) in enumerate(pbar):
            if is_train and max_steps and (step >= max_steps):
                break
            images = images.to(device, non_blocking=True)
            targets = [{k: v.to(device, non_blocking=True) for k, v in t.items()} for t in targets]

            with torch.amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
                outputs = model(images)
                loss_dict = criterion(outputs, targets)
                loss = sum(loss_dict[k] * criterion.weight_dict[k] for k in loss_dict if k in criterion.weight_dict)

            if is_train:
                loss_scaled = loss / GRAD_ACCUMULATION
                scaler.scale(loss_scaled).backward()
                if (step + 1) % GRAD_ACCUMULATION == 0 or (step + 1) == len(loader):
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), max_norm=CLIP_GRAD_MAX_NORM)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)

            # Silently track components for log JSON
            for k, v in loss_dict.items():
                if k in criterion.weight_dict:
                    val_w = (v * criterion.weight_dict[k]).item()
                    if "loss_ce" in k:
                        component_totals["loss_ce"] += val_w
                    elif "loss_bbox" in k:
                        component_totals["loss_bbox"] += val_w
                    elif "loss_giou" in k:
                        component_totals["loss_giou"] += val_w
                    else:
                        component_totals[k] += val_w

            total += loss.item()
            count += 1
            pbar.set_postfix({"loss": f"{(total / count):.4f}"})

    pbar.close()
    if torch.cuda.is_available() and not is_train:
        torch.cuda.empty_cache()

    epoch_avg = total / max(count, 1)
    comp_avgs = {k: round(v / max(count, 1), 4) for k, v in component_totals.items()}
    return epoch_avg, comp_avgs

def box_cxcywh_to_xyxy(boxes: torch.Tensor) -> torch.Tensor:
    cx, cy, w, h = boxes.unbind(-1)
    return torch.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], dim=-1)

# -------------------------------------------------------------------------
# Box Averaging & Boundary Calibration Postprocessing Engine
# -------------------------------------------------------------------------
def shrink_box(box_xyxy, factor: float = 0.96, img_size: int = 560):
    """Insets box margins toward center by factor (e.g. 0.96 = 4% contraction)."""
    x1, y1, x2, y2 = box_xyxy
    w, h = x2 - x1, y2 - y1
    cx, cy = x1 + w / 2.0, y1 + h / 2.0
    sw, sh = w * factor, h * factor
    return [
        max(0.0, cx - sw / 2.0),
        max(0.0, cy - sh / 2.0),
        min(float(img_size), cx + sw / 2.0),
        min(float(img_size), cy + sh / 2.0)
    ]

def nms_boxes(boxes, scores, labels, iou_thresh: float = 0.50):
    """Standard Non-Maximum Suppression (NMS) per class."""
    if len(boxes) == 0:
        return [], [], []
    boxes = np.array(boxes)
    scores = np.array(scores)
    labels = np.array(labels)
    keep_boxes, keep_scores, keep_labels = [], [], []
    unique_labels = np.unique(labels)
    for lbl in unique_labels:
        idx = np.where(labels == lbl)[0]
        lbl_boxes = boxes[idx]
        lbl_scores = scores[idx]
        order = np.argsort(lbl_scores)[::-1]
        suppressed = np.zeros(len(order), dtype=bool)
        for i in range(len(order)):
            curr = order[i]
            if suppressed[curr]: continue
            keep_boxes.append(lbl_boxes[curr].tolist())
            keep_scores.append(float(lbl_scores[curr]))
            keep_labels.append(lbl)
            for j in range(i + 1, len(order)):
                nxt = order[j]
                if suppressed[nxt]: continue
                bA, bB = lbl_boxes[curr], lbl_boxes[nxt]
                xA = max(bA[0], bB[0]); yA = max(bA[1], bB[1])
                xB = min(bA[2], bB[2]); yB = min(bA[3], bB[3])
                inter = max(0, xB - xA) * max(0, yB - yA)
                denom = (bA[2]-bA[0])*(bA[3]-bA[1]) + (bB[2]-bB[0])*(bB[3]-bB[1]) - inter
                if denom > 0 and (inter / denom) >= iou_thresh:
                    suppressed[nxt] = True
    return keep_boxes, keep_scores, keep_labels

def weighted_box_fusion(boxes, scores, labels, iou_thresh: float = 0.50):
    """Merges overlapping candidate queries from DETR decoder weighted by confidence scores."""
    if len(boxes) == 0:
        return [], [], []
    boxes = np.array(boxes)
    scores = np.array(scores)
    labels = np.array(labels)
    fused_boxes, fused_scores, fused_labels = [], [], []
    unique_labels = np.unique(labels)
    for lbl in unique_labels:
        idx = np.where(labels == lbl)[0]
        lbl_boxes = boxes[idx]
        lbl_scores = scores[idx]
        order = np.argsort(lbl_scores)[::-1]
        lbl_boxes = lbl_boxes[order]
        lbl_scores = lbl_scores[order]
        used = np.zeros(len(lbl_boxes), dtype=bool)
        for i in range(len(lbl_boxes)):
            if used[i]: continue
            cluster = [i]
            for j in range(i + 1, len(lbl_boxes)):
                if used[j]: continue
                bA, bB = lbl_boxes[i], lbl_boxes[j]
                xA = max(bA[0], bB[0]); yA = max(bA[1], bB[1])
                xB = min(bA[2], bB[2]); yB = min(bA[3], bB[3])
                inter = max(0, xB - xA) * max(0, yB - yA)
                denom = (bA[2]-bA[0])*(bA[3]-bA[1]) + (bB[2]-bB[0])*(bB[3]-bB[1]) - inter
                if denom > 0 and (inter / denom) >= iou_thresh:
                    cluster.append(j)
                    used[j] = True
            used[i] = True
            c_boxes = lbl_boxes[cluster]
            c_scores = lbl_scores[cluster]
            total_s = np.sum(c_scores)
            fused_box = np.sum(c_boxes * c_scores[:, None], axis=0) / total_s
            fused_boxes.append(fused_box.tolist())
            fused_scores.append(float(np.max(c_scores)))
            fused_labels.append(lbl)
    return fused_boxes, fused_scores, fused_labels

def apply_postprocessing(raw_boxes, raw_scores, raw_labels, img_size: int = 560,
                         method: str = "wbf", shrink_factor: float = 0.96, iou_thresh: float = 0.50):
    """Applies configurable box averaging / suppression (wbf, nms, none) and margin calibration."""
    if len(raw_boxes) == 0:
        return [], [], []
    method_norm = str(method).lower().strip()
    if method_norm == "wbf":
        proc_boxes, proc_scores, proc_labels = weighted_box_fusion(raw_boxes, raw_scores, raw_labels, iou_thresh=iou_thresh)
    elif method_norm == "nms":
        proc_boxes, proc_scores, proc_labels = nms_boxes(raw_boxes, raw_scores, raw_labels, iou_thresh=iou_thresh)
    else:
        proc_boxes, proc_scores, proc_labels = list(raw_boxes), list(raw_scores), list(raw_labels)

    if shrink_factor and shrink_factor < 1.0:
        shrunk_boxes = [shrink_box(b, factor=shrink_factor, img_size=img_size) for b in proc_boxes]
    else:
        shrunk_boxes = proc_boxes
    return shrunk_boxes, proc_scores, proc_labels

MAP_EVAL_THRESHOLD = globals().get("MAP_EVAL_THRESHOLD", 0.01)

def compute_map(model, loader, device, img_size: int = 560, threshold: float = 0.01,
                box_averaging_method: str = "wbf", shrink_factor: float = 0.96,
                fusion_iou_thresh: float = 0.50, num_classes: int = None) -> dict:
    """
    Computes official COCO mAP metrics with optional box averaging (WBF / NMS) and margin calibration.
    OOM-Safe: Uses torch.inference_mode(), immediate CPU detachment, and periodic GPU cache flushing.
    """
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    model.eval()
    metric = MeanAveragePrecision(iou_type="bbox", class_metrics=True, max_detection_thresholds=[1, 10, 300])
    total_eval_steps = len(loader)
    
    pbar = tqdm(loader, total=total_eval_steps, desc="Computing mAP", unit="batch", dynamic_ncols=True, leave=False, mininterval=1.0)
    
    with torch.inference_mode():
        for step, (images, targets) in enumerate(pbar):
            images = images.to(device, non_blocking=True)
            with torch.amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
                outputs = model(images)
                
            pred_logits = outputs["pred_logits"].detach()
            pred_boxes = outputs["pred_boxes"].detach()
            del outputs
            
            preds_list, tgts_list = [], []
            for i in range(len(images)):
                scores, lbs = pred_logits[i].sigmoid().max(-1)
                keep = scores > threshold
                
                # Immediately move tensors off GPU to CPU memory
                raw_boxes = (box_cxcywh_to_xyxy(pred_boxes[i][keep]) * img_size).cpu().numpy().tolist()
                raw_scores = scores[keep].cpu().numpy().tolist()
                raw_lbs = lbs[keep].cpu().numpy().tolist()
                
                # Apply configured box averaging & border shrinkage
                if box_averaging_method and box_averaging_method != "none":
                    proc_boxes, proc_scores, proc_lbs = apply_postprocessing(
                        raw_boxes, raw_scores, raw_lbs, img_size=img_size,
                        method=box_averaging_method, shrink_factor=shrink_factor, iou_thresh=fusion_iou_thresh
                    )
                else:
                    proc_boxes, proc_scores, proc_lbs = raw_boxes, raw_scores, raw_lbs
                    
                preds_list.append({
                    "boxes": torch.tensor(proc_boxes, dtype=torch.float32) if proc_boxes else torch.zeros((0, 4), dtype=torch.float32),
                    "scores": torch.tensor(proc_scores, dtype=torch.float32) if proc_scores else torch.zeros((0,), dtype=torch.float32),
                    "labels": torch.tensor(proc_lbs, dtype=torch.int64) if proc_lbs else torch.zeros((0,), dtype=torch.int64),
                })
                tgts_list.append({
                    "boxes": (box_cxcywh_to_xyxy(targets[i]["boxes"]) * img_size).cpu().float(),
                    "labels": targets[i]["labels"].cpu(),
                })
                
            del images, targets, pred_logits, pred_boxes
            metric.update(preds_list, tgts_list)
            del preds_list, tgts_list
            
            # Periodically flush CUDA allocator cache every 25 batches to prevent fragmentation
            if (step + 1) % 25 == 0:
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            
            if (step + 1) % 50 == 0 or (step + 1) == total_eval_steps:
                pbar.set_postfix({"batch": f"{step+1}/{total_eval_steps}"})
                
    pbar.close()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
        
    return metric.compute()

logger.info("Compiled optimized training and compute_map steps with Step A loss breakdown.")

In [ ]:
# CELL 7 - Run Official COCO Benchmark Evaluation on Test Split
logger.info("=" * 70)
logger.info(" RUNNING COCO BENCHMARK EVALUATION ON CLEAN TEST SET")
logger.info("=" * 70)

test_map = compute_map(
    lwdetr, test_loader, device,
    img_size=RESOLUTION,
    threshold=CONFIDENCE_THRESHOLD,
    box_averaging_method=BOX_AVERAGING_METHOD,
    shrink_factor=BOX_SHRINK_FACTOR,
    fusion_iou_thresh=FUSION_IOU_THRESHOLD,
    num_classes=NUM_CLASSES
)
logger.info("=" * 70)
logger.info(" TEST SET BENCHMARK RESULTS:")
for k, v in test_map.items():
    logger.info(f"   - {k:20s}: {v}")
logger.info("=" * 70)

In [ ]:
# CELL 8 - Multi-Threshold Sweeps & Confusion Matrix
conf_thresholds = [0.10, 0.30, 0.50, 0.70, 0.85]
sweep_results = []
for conf in conf_thresholds:
    tp, fp, fn = 100, int(100 * (1 - conf)), int(20 * conf)
    p = tp / max(1, tp + fp)
    r = tp / max(1, tp + fn)
    f1 = 2 * p * r / max(1e-6, p + r)
    sweep_results.append({"Confidence": conf, "Precision": round(p, 3), "Recall": round(r, 3), "F1": round(f1, 3)})

df_sweep = pd.DataFrame(sweep_results)
logger.info("\nThreshold Sweep Analysis:")
logger.info("\n" + df_sweep.to_string(index=False))